# OEC AudioGen — Gradio client for the model server

**UI environment:** \`phase1\` (Gradio + requests only for this notebook).  
**Model server:** AudioCraft 1.3.0 / PyTorch 2.1 + CUDA 11.8 at \`http://127.0.0.1:7874\`.  
The UI does not load PyTorch or model weights. Start the server from the OEC repo root on a genuinely idle A100:

\`\`\`bash
python tools/oec_service_manager.py start audiocraft-api --provider audiogen --gpu <ID> --port 7874 --allow-download --accept-noncommercial-license
python tools/oec_service_manager.py status audiocraft-api
\`\`\`

Use \`--allow-download\` only when a cache fill is needed and permitted by current OEC StorageManager admission. AudioGen model weights have a CC-BY-NC 4.0 use boundary; confirm that the intended use is permitted. The API binds to loopback; use an SSH tunnel for remote browser access. Public Gradio sharing stays disabled.

import hashlib
import json
import os
from pathlib import Path
import time
import urllib.parse
import wave
import gradio as gr
import requests

API_BASE = os.environ.get("OEC_MEDIA_API_URL", "http://127.0.0.1:7874").rstrip("/")
RESULT_ROOT = Path("/tmp") / "oec-notebook-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if urllib.parse.urlsplit(API_BASE).hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise RuntimeError("AudioGen notebook only permits a loopback OEC media API endpoint.")
HTTP = requests.Session()
HTTP.trust_env = False

def service_health():
    try:
        response = HTTP.get(f"{API_BASE}/health", timeout=8)
        payload = response.json()
    except Exception as exc:
        raise gr.Error(
            f"AudioGen model server is unreachable at {API_BASE}. "
            "Start the approved audiocraft-api service and check its logs."
        ) from exc
    if response.status_code != 200 or not payload.get("ready") or not payload.get("model_loaded"):
        raise gr.Error(f"AudioGen model server is not ready: {json.dumps(payload, ensure_ascii=False)}")
    return payload

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def submit_and_wait(payload, timeout_seconds=900):
    health = service_health()
    try:
        response = HTTP.post(f"{API_BASE}/v1/jobs", json=payload, timeout=30)
    except Exception as exc:
        raise gr.Error(f"AudioGen job submission failed: {type(exc).__name__}") from exc
    if response.status_code != 202:
        raise gr.Error(f"AudioGen job was rejected (HTTP {response.status_code}): {response.text[:1200]}")
    accepted = response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected AudioGen job response: {accepted}")
    deadline = time.monotonic() + timeout_seconds
    latest = accepted
    while time.monotonic() < deadline:
        poll = HTTP.get(f"{API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"AudioGen job status failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"AudioGen job {status}: {latest.get('message', 'No error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error("AudioGen generation timed out; inspect the audiocraft-api service logs.")
    result = HTTP.get(f"{API_BASE}/v1/jobs/{job_id}/result", timeout=120)
    if result.status_code != 200:
        raise gr.Error(f"AudioGen result download failed (HTTP {result.status_code}): {result.text[:800]}")
    content_type = result.headers.get("Content-Type", "").split(";", 1)[0].lower()
    if content_type != "audio/wav":
        raise gr.Error(f"Expected a WAV result, received Content-Type={content_type!r}.")
    output_path = RESULT_ROOT / f"{job_id}.wav"
    if output_path.exists():
        raise gr.Error("Refusing to overwrite an existing notebook result.")
    output_path.write_bytes(result.content)
    try:
        with wave.open(str(output_path), "rb") as audio:
            frames, rate, channels = audio.getnframes(), audio.getframerate(), audio.getnchannels()
    except Exception as exc:
        output_path.unlink(missing_ok=True)
        raise gr.Error("AudioGen server returned a WAV that could not be decoded.") from exc
    if output_path.stat().st_size < 64 or frames <= 0 or rate <= 0 or channels <= 0:
        output_path.unlink(missing_ok=True)
        raise gr.Error("AudioGen server returned an empty or invalid WAV.")
    details = {
        "job_id": job_id, "status": latest.get("status"), "api": API_BASE,
        "health": health, "metadata": latest.get("metadata", {}),
        "artifact_path": str(output_path), "artifact_bytes": output_path.stat().st_size,
        "artifact_sha256": sha256_file(output_path),
        "sample_rate_hz": rate, "channels": channels, "duration_seconds": frames / rate,
    }
    return str(output_path), json.dumps(details, indent=2, ensure_ascii=False)

In [ ]:
def generate_sound_effect(prompt, duration, seed):
    if not isinstance(prompt, str) or not prompt.strip():
        raise gr.Error("Enter a sound-effect description.")
    if not 1 <= int(duration) <= 10:
        raise gr.Error("Duration must be between 1 and 10 seconds.")
    seed = int(seed)
    if not 0 <= seed <= 2**31 - 1:
        raise gr.Error("Seed must be between 0 and 2147483647.")
    payload = {
        "task": "text_to_audio",
        "prompt": prompt.strip(),
        "parameters": {"duration_seconds": int(duration), "seed": seed},
    }
    return submit_and_wait(payload)

demo = gr.Interface(
    fn=generate_sound_effect,
    inputs=[
        gr.Textbox(label="Sound-effect prompt", lines=3,
                   value="Short cinematic rain shower with distant thunder"),
        gr.Slider(1, 10, value=3, step=1, label="Duration (seconds)"),
        gr.Number(value=42, precision=0, label="Seed"),
    ],
    outputs=[
        gr.Audio(type="filepath", label="Generated WAV"),
        gr.Textbox(label="Server and artifact metadata", lines=12),
    ],
    title="OEC AudioGen — model server",
    description="Generation runs in the isolated AudioCraft backend. This page only submits a job and validates the returned WAV.",
)
if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo.queue(max_size=1).launch(
        server_name="127.0.0.1", server_port=7901, share=False, inline=False, show_error=True
    )